# Baselines comparison - CIFAR-10
Compares Random, Uncertainty, Margin, Entropy, CoreSet, BADGE, BALD, DBAL, TPCDC, TPCRP across budgets

### config

In [ ]:
BUDGETS = [10, 20, 30, 40, 50, 60, 100, 150, 200, 250, 300]

N_REPEATS = 10
CLF_EPOCHS = 200
CLF_LR = 0.025
WEIGHT_DECAY = 5e-4
K_NEIGHBOURS = 20
MAX_CLUSTERS = 500
MIN_CLUSTER_SIZE = 5
SEED = 42

MC_DROPOUT_P = 0.5
MC_T = 10

BADGE_SUBSAMPLE = 5000

DATA_DIR = "./data" #CIFAR 10 data 
EMB_PATH = "../embeddings/embeddings_500epochs.npy" 
LBL_PATH = "../labels/labels.npy"
SCAN_DIR = "../scan_assignments"


### imports

In [ ]:
import os, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

torch.manual_seed(SEED)
np.random.seed(SEED)

# Machines or Kaggle or Colab
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")


CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR_STD = [0.2023, 0.1994, 0.2010]
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

print("device: ", DEVICE)


### data

In [ ]:
train_transform = T.Compose(
    [
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
)
test_transform = T.Compose(
    [
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
)

full_train = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=True, download=True, transform=train_transform
)
test_set = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=False, download=True, transform=test_transform
)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=512, shuffle=False, num_workers=0)

plain_transform = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
dataset_plain = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=True, download=False, transform=plain_transform
)

embeddings = np.load(EMB_PATH)
labels = np.load(LBL_PATH)

print("train:", len(full_train), "test:", len(test_set))
print("embeddings shape:", embeddings.shape)


### helpers

In [ ]:

class LabeledSubset(torch.utils.data.Dataset):
    def __init__(self, base, indices):
        self.base, self.indices = base, indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return self.base[self.indices[i]]

def train_and_eval(selected_indices, epochs=CLF_EPOCHS, lr=CLF_LR, wd=WEIGHT_DECAY, seed=SEED):
    torch.manual_seed(seed)
    subset = LabeledSubset(full_train, selected_indices)
    loader = torch.utils.data.DataLoader(
        subset, batch_size=min(128, len(subset)), shuffle=True, num_workers=0, drop_last=False
    )

    net = torchvision.models.resnet18(weights=None)
    net.fc = nn.Linear(net.fc.in_features, 10)
    net = net.to(DEVICE)

    optim = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    crit = nn.CrossEntropyLoss()

    for _ in range(epochs):
        net.train()
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optim.zero_grad()
            crit(net(imgs), lbls).backward()
            optim.step()
        sched.step()

    net.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            correct += (net(imgs).argmax(1) == lbls).sum().item()
            total += lbls.size(0)
    return 100.0 * correct / total, net

@torch.no_grad()
def get_softmax_all(net):
    plain_transform = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
    plain_set = torchvision.datasets.CIFAR10(
        root=DATA_DIR, train=True, download=False, transform=plain_transform
    )
    plain_loader = torch.utils.data.DataLoader(
        plain_set, batch_size=512, shuffle=False, num_workers=0
    )
    net.eval()
    probs = []
    for imgs, _ in plain_loader:
        p = F.softmax(net(imgs.to(DEVICE)), dim=1).cpu().numpy()
        probs.append(p)
    return np.concatenate(probs)

print("helpers ready")


### query strategies

In [ ]:
# Random
def strategy_random(embeddings, budget, model=None, seed=SEED, **kwargs):
    rng = np.random.default_rng(seed)
    return rng.choice(len(embeddings), budget, replace=False)

# Uncertainty
def strategy_uncertainty(embeddings, budget, model=None, seed=SEED, **kwargs):
    if model is None:

        return strategy_random(embeddings, budget, seed=seed)
    probs = get_softmax_all(model)
    scores = 1.0 - probs.max(axis=1)
    return np.argsort(-scores)[:budget]

# Margin
def strategy_margin(embeddings, budget, model=None, seed=SEED, **kwargs):
    if model is None:
        return strategy_random(embeddings, budget, seed=seed)
    probs = get_softmax_all(model)
    sorted_p = np.sort(probs, axis=1)[:, ::-1]
    margin = sorted_p[:, 0] - sorted_p[:, 1]
    return np.argsort(margin)[:budget]

# Entropy
def strategy_entropy(embeddings, budget, model=None, seed=SEED, **kwargs):
    if model is None:
        return strategy_random(embeddings, budget, seed=seed)
    probs = get_softmax_all(model)

    probs = np.clip(probs, 1e-8, 1.0)
    entropy = -(probs * np.log(probs)).sum(axis=1)
    return np.argsort(-entropy)[:budget]

# CoreSet — greedy, slow but works
def strategy_coreset(embeddings, budget, model=None, seed=SEED, **kwargs):
    rng = np.random.default_rng(seed)
    N = len(embeddings)
    emb = embeddings.astype(np.float32)

    selected = [int(rng.integers(0, N))]

    min_dist = np.full(N, np.inf)

    for _ in range(budget - 1):

        last = emb[selected[-1]]
        dists = np.linalg.norm(emb - last, axis=1)
        min_dist = np.minimum(min_dist, dists)

        next_idx = int(np.argmax(min_dist))
        selected.append(next_idx)
        min_dist[next_idx] = 0.0

    return np.array(selected)

# TPCRP — the actual paper method
def strategy_tpcrp(embeddings, budget, model=None, seed=SEED, global_typicality=None, **kwargs):
    K = min(budget, MAX_CLUSTERS)
    if K <= 50:
        km = KMeans(n_clusters=K, random_state=seed, n_init=10)
    else:
        km = MiniBatchKMeans(n_clusters=K, random_state=seed, n_init=5, batch_size=2048)
    assignments = km.fit_predict(embeddings)
    sizes = np.bincount(assignments, minlength=K)
    sorted_clusters = np.argsort(-sizes)

    selected = []
    for cid in sorted_clusters:
        if len(selected) >= budget:
            break
        c_idxs = np.where(assignments == cid)[0]
        if len(c_idxs) < MIN_CLUSTER_SIZE:
            continue
        best = c_idxs[np.argmax(global_typicality[c_idxs])]
        selected.append(int(best))
    return np.array(selected)


In [ ]:

class MCDropoutWrapper(nn.Module):
    def __init__(self, base_model, dropout_p=0.5):
        super().__init__()
        children = list(base_model.children())
        self.backbone = nn.Sequential(*children[:-1])
        self.fc = children[-1]
        self.dropout = nn.Dropout(p=dropout_p)

    def forward(self, x):
        h = self.backbone(x).flatten(1)
        return self.fc(self.dropout(h))

def kmeans_pp_select(X, budget, seed=SEED):
    rng = np.random.default_rng(seed)
    N = len(X)
    sel = [int(rng.integers(0, N))]
    min_d = np.full(N, np.inf)
    for _ in range(budget - 1):
        d = np.sum((X - X[sel[-1]]) ** 2, axis=1)
        min_d = np.minimum(min_d, d)
        prob = min_d / min_d.sum()
        sel.append(int(rng.choice(N, p=prob)))
    return np.array(sel)

@torch.no_grad()
def get_gradient_embeddings(model, dataset_plain, subsample_idx):
    model.eval()
    feats_buf = []

    def _hook(module, inp, out):
        feats_buf.append(inp[0].detach().cpu())

    handle = model.fc.register_forward_hook(_hook)
    loader = torch.utils.data.DataLoader(
        LabeledSubset(dataset_plain, subsample_idx), batch_size=512, shuffle=False, num_workers=0
    )
    probs_list = []
    for imgs, _ in loader:
        logits = model(imgs.to(DEVICE))
        probs_list.append(F.softmax(logits, dim=1).cpu().numpy())
    handle.remove()
    feats = torch.cat(feats_buf).numpy()
    probs = np.concatenate(probs_list)
    M, D = feats.shape
    C = probs.shape[1]
    pred_y = probs.argmax(axis=1)
    one_hot = np.zeros_like(probs)
    one_hot[np.arange(M), pred_y] = 1.0
    diff = probs - one_hot
    return (diff[:, :, None] * feats[:, None, :]).reshape(M, C * D)

def get_mc_probs(mc_model, dataset_plain, T=10):
    mc_model.train()
    loader = torch.utils.data.DataLoader(
        dataset_plain, batch_size=512, shuffle=False, num_workers=0
    )
    passes = []
    for _ in range(T):
        p_t = []
        with torch.no_grad():
            for imgs, _ in loader:
                p = F.softmax(mc_model(imgs.to(DEVICE)), dim=1).cpu().numpy()
                p_t.append(p)
        passes.append(np.concatenate(p_t))
    return np.stack(passes, axis=1)

# BADGE
def strategy_badge(embeddings, budget, model=None, seed=SEED, dataset_plain=None, **kwargs):
    if model is None or dataset_plain is None:
        return strategy_random(embeddings, budget, seed=seed)
    N = len(embeddings)
    rng = np.random.default_rng(seed)
    subsample = rng.choice(N, min(N, BADGE_SUBSAMPLE), replace=False)
    grad_emb = get_gradient_embeddings(model, dataset_plain, subsample)
    local = kmeans_pp_select(grad_emb, budget, seed=seed)
    return subsample[local]

# BALD
def strategy_bald(embeddings, budget, mc_probs=None, seed=SEED, **kwargs):
    if mc_probs is None:
        return strategy_random(embeddings, budget, seed=seed)
    mean_p = np.clip(mc_probs.mean(axis=1), 1e-8, 1.0)
    H_mean = -(mean_p * np.log(mean_p)).sum(axis=1)
    p_clip = np.clip(mc_probs, 1e-8, 1.0)
    H_each = -(p_clip * np.log(p_clip)).sum(axis=2)
    bald = H_mean - H_each.mean(axis=1)
    return np.argsort(-bald)[:budget]

# DBAL
def strategy_dbal(embeddings, budget, mc_probs=None, seed=SEED, **kwargs):
    if mc_probs is None:
        return strategy_random(embeddings, budget, seed=seed)
    mean_p = np.clip(mc_probs.mean(axis=1), 1e-8, 1.0)
    entropy = -(mean_p * np.log(mean_p)).sum(axis=1)
    return np.argsort(-entropy)[:budget]

# TPCDC
def strategy_tpcdc(embeddings, budget, seed=SEED, global_typicality=None, **kwargs):
    K = min(budget, MAX_CLUSTERS)
    scan_path = f"{SCAN_DIR}/scan_K{K}.npy"
    if os.path.exists(scan_path):
        assignments = np.load(scan_path)
    else:
        import warnings

        warnings.warn(f"[TPCDC] SCAN file not found: {scan_path} — falling back to k-means")
        if K <= 50:
            km = KMeans(n_clusters=K, random_state=seed, n_init=10)
        else:
            km = MiniBatchKMeans(n_clusters=K, random_state=seed, n_init=5, batch_size=2048)
        assignments = km.fit_predict(embeddings)
    sizes = np.bincount(assignments, minlength=K)
    sorted_clusters = np.argsort(-sizes)
    selected = []
    for cid in sorted_clusters:
        if len(selected) >= budget:
            break
        c_idxs = np.where(assignments == cid)[0]
        if len(c_idxs) < MIN_CLUSTER_SIZE:
            continue
        best = c_idxs[np.argmax(global_typicality[c_idxs])]
        selected.append(int(best))
    return np.array(selected)

print("all strategies ready")

STRATEGIES = {"Random":strategy_random,
    "Uncertainty":strategy_uncertainty,
    "Margin":strategy_margin,
    "Entropy":strategy_entropy,
    "CoreSet":strategy_coreset,
    "BADGE":strategy_badge,
    "BALD":strategy_bald,
    "DBAL":strategy_dbal,
    "TPCDC":strategy_tpcdc,
    "TPCRP":strategy_tpcrp,}

print(f"loaded {len(STRATEGIES)} strategies")


### global typicality

In [ ]:
print("Computing typicality...")

t0 = time.time()
nbrs = NearestNeighbors(
    n_neighbors=K_NEIGHBOURS + 1, algorithm="auto", metric="euclidean", n_jobs=-1
)

nbrs.fit(embeddings)
dists, _ = nbrs.kneighbors(embeddings)
GLOBAL_TYPICALITY = 1.0 / (dists[:, 1:].mean(axis=1) + 1e-8)

print(f"done in {time.time()-start:.1f}s")


### sweep

In [ ]:
results = {name: {b:[] for b in BUDGETS} for name in STRATEGIES}

UNCERTAINTY_STRATEGIES = {"Uncertainty", "Margin", "Entropy"}
BAYESIAN_STRATEGIES = {"BALD", "DBAL"}

for budget in BUDGETS:
    print(f"budget {budget}")

    for rep in range(N_REPEATS):
        seed_i = SEED + rep * 100  # different seed each rep
        np.random.seed(seed_i)
        torch.manual_seed(seed_i)

        init_idx = np.random.choice(len(embeddings), budget, replace=False) # warm start
        _, warm_model = train_and_eval(init_idx, seed=seed_i)

        _mc_wrap = MCDropoutWrapper(warm_model, MC_DROPOUT_P).to(DEVICE)
        mc_probs_cache = get_mc_probs(_mc_wrap, dataset_plain, T=MC_T)

        for name, strategy_fn in STRATEGIES.items():
            idx = strategy_fn(
                embeddings,
                budget,
                model=warm_model,
                mc_probs=mc_probs_cache,
                seed=rep_seed,
                global_typicality=GLOBAL_TYPICALITY,
                dataset_plain=dataset_plain,
            )
            acc, _ = train_and_eval(idx, seed=seed_i)
            results[name][budget].append(acc)

        accs = "  ".join(f"{n}={results [n ][budget ][-1 ]:.1f}%" for n in STRATEGIES)
        print(f"  rep {rep+1}: {accs}")

print("all done!")


### results

In [ ]:

names = list(STRATEGIES.keys())
print("budget  " + "  ".join(f"{n:>12}" for n in names))


for b in BUDGETS:
    row = f"B={b:<4}  "
    for n in names:
        m = np.mean(results[n][b])
        row += f"  {m:5.2f}%    "
    print(row)


### plots

In [ ]:

BUDGETS_LOW = [10, 20, 30, 40, 50, 60]
BUDGETS_HIGH = [50, 100, 150, 200, 250, 300]

names = list(STRATEGIES.keys())

colors = {"Random":"#7F8C8D",
    "Uncertainty":"#E67E22",
    "Margin":"#9B59B6",
    "Entropy":"#1ABC9C",
    "CoreSet":"#2980B9",
    "BADGE":"#F39C12",
    "BALD":"#8E44AD",
    "DBAL":"#16A085",
    "TPCDC":"#E74C3C",
    "TPCRP":"#C0392B",}
styles = {"Random":("s", "--"),
    "Uncertainty":("^", "--"),
    "Margin":("D", "--"),
    "Entropy":("v", "--"),
    "CoreSet":("P", "-"),
    "BADGE":("h", "--"),
    "BALD":("8", "--"),
    "DBAL":("*", "--"),
    "TPCDC":("X", "-"),
    "TPCRP":("o", "-"),}
TYPICLUST = {"TPCRP", "TPCDC"}

means = {n: {b:np.mean(results[n][b]) for b in BUDGETS} for n in names}
stds = {n: {b:np.std(results[n][b]) for b in BUDGETS} for n in names}

def plot_baselines(budget_range, filename, title_suffix):
    fig, ax = plt.subplots(figsize=(10, 5))
    for n in names:
        m = [means[n][b] for b in budget_range]
        s = [stds[n][b] for b in budget_range]
        marker, ls = styles[n]
        lw = 2.4 if n in TYPICLUST else 1.4
        ax.errorbar(
            budget_range,
            m,
            yerr=s,
            fmt=f"{marker}{ls}",
            color=colors[n],
            linewidth=lw,
            capsize=3,
            markersize=5.5,
            label=n,
            zorder=10 if n in TYPICLUST else 3,
        )

    ax.set_xlabel("Budget B (total labeled examples)", fontsize=11)
    ax.set_ylabel("Test accuracy (%)", fontsize=11)
    ax.set_title(f"Active Learning Strategies — CIFAR-10\n{title_suffix}", fontsize=11)
    ax.set_xticks(budget_range)
    ax.set_xticklabels([f"{b}\n({b//10}/cls)" for b in budget_range], fontsize=8.5)
    ax.legend(fontsize=8.5, framealpha=0.9, loc="upper left", ncol=2)
    ax.grid(axis="y", alpha=0.25, linewidth=0.6)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {filename}")

plot_baselines(
    BUDGETS_HIGH, "./baselines_high_budget_vv.png", "Higher budget B=[50-300] (5-30 labels/class)"
)



### summary

In [ ]:
names = list(STRATEGIES.keys())
print("average accuracy per strategy:")

for n in names:
    avg = np.mean([np.mean(results[n][b]) for b in BUDGETS])
    print(f"  {n}: {avg:.2f}%")
    
tpcrp_avg = np.mean([np.mean(results["TPCRP"][b]) for b in BUDGETS])
random_avg = np.mean([np.mean(results["Random"][b]) for b in BUDGETS])


print(f"TPCRP gain over Random: +{tpcrp_avg-random_avg:.2f}%")
